# Phase VII — targeted convergence sensitivity check

This notebook reruns only **ArtBench-10 All10 / B90+G44 / outer fold 1**, the one fold that reached `max_iter=5000` in the final fixed linear probe. It preserves the frozen data, artist-disjoint split, preprocessing, `C=1`, `dual=False`, and `tol=1e-3`, but increases `max_iter` to 20,000 and compares predictions against the saved Phase VII checkpoint.

PASS criterion: the refit converges and `abs(delta macro-F1) < 0.001`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, pathlib
REPO = pathlib.Path('/content/painting-geometry')
if REPO.exists():
    subprocess.run(['rm','-rf',str(REPO)], check=True)
subprocess.run(['git','clone','--branch','multiscale-corpus-analysis','--single-branch','https://github.com/ardominguezm/painting-geometry.git',str(REPO)], check=True)
os.chdir(REPO)
commit = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
print('Repository commit:', commit)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scikit-learn'], check=True)

In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/painting_geometry_phase7_full')
FEATURES = ROOT/'results'/'artbench_full_features_with_ordinal.csv'
CKPT = ROOT/'results'/'phase7_fixed_linear_probe'/'artbench10_all'/'_representation_checkpoints'/'B90_G44.npz'
OUT = ROOT/'results'/'phase7_fixed_linear_probe'/'convergence_sensitivity'
print('Features:', FEATURES, FEATURES.exists())
print('Checkpoint:', CKPT, CKPT.exists())
assert FEATURES.exists(), FEATURES
assert CKPT.exists(), CKPT

In [ ]:
cmd = [
    sys.executable, '-u', 'scripts/check_phase7_b90_g44_convergence.py',
    '--features', str(FEATURES),
    '--checkpoint', str(CKPT),
    '--output-dir', str(OUT),
    '--fold', '1',
    '--max-iter', '20000',
]
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
result = pd.read_csv(OUT/'b90_g44_fold1_convergence_sensitivity.csv')
display(result.T)

If the final line reports **`VERDICT: PASS ✓`**, the previously reported Phase VII B90+G44 result can be retained as numerically stable. The sensitivity CSV/JSON remains permanently stored in Google Drive under `results/phase7_fixed_linear_probe/convergence_sensitivity/`.